# Laboratorium 4: Tuning i ewaluacja sieci neuronowych

Na CIFAR-10 (jak w laboratorium 2) uczycie się kontrolować trening callbackami (EarlyStopping, ModelCheckpoint, ReduceLROnPlateau), porównujecie optymalizatory i szukacie hiperparametrów KerasTunerem. Wybrany na walidacji model oceniacie raz na zbiorze testowym: macierz pomyłek, `classification_report`, najpewniejsze błędy. Na koniec sprawdzacie metodą Grad-CAM, na które fragmenty obrazu patrzy sieć, najpierw na własnej CNN, a potem na modelu MobileNetV2 z laboratorium 3. Zadania rozszerzone porównują Hyperband z RandomSearch i mierzy, jak bardzo wynik zależy od ziarna losowania.

Środowisko: Google Colab z GPU (Środowisko wykonawcze, Zmień typ środowiska wykonawczego, T4 GPU). Keras 3 na backendzie TensorFlow. Pierwsza komórka kodu instaluje KerasTuner.

In [ ]:
# KerasTuner nie jest preinstalowany w Colabie. Z backendem TensorFlow KerasTuner importuje też tensorboard,
# więc instalujemy oba pakiety. Jeśli są już zainstalowane, komórka tylko to potwierdzi.
%pip install -q keras-tuner tensorboard

## Konfiguracja

Wszystkie rozmiary i budżety treningu są w jednej komórce. `TRYB_SKROCONY = False` to wartości dla Colaba z GPU (czasu w Colabie nie mierzono). `TRYB_SKROCONY = True` to wersja zredukowana, w której wykonano rozwiązanie wzorcowe na CPU. Jeśli w Colabie treningi trwają za długo, zmniejszcie `N_TRAIN`, `MAX_PROB` albo liczby epok i zapiszcie we wnioskach, co zmieniliście.

In [ ]:
# Konfiguracja przebiegu. False: wartości dla Colaba z GPU T4 (czasu w Colabie nie mierzono).
# True: wersja zredukowana, w której wykonano rozwiązanie wzorcowe lokalnie na CPU.
TRYB_SKROCONY = False

N_VAL = 5_000                                # walidacja: ostatnie N_VAL obrazów części treningowej CIFAR-10
BATCH = 64
N_TRAIN = 10_000 if TRYB_SKROCONY else 20_000   # obrazy treningowe (pierwsze N_TRAIN)
EPOKI_MAX = 30 if TRYB_SKROCONY else 40          # górna granica epok przy EarlyStopping (zadania 1 i 3)
EPOKI_OPT = 10 if TRYB_SKROCONY else 15          # porównanie optymalizatorów (zadanie 2)
MAX_PROB = 8                                     # liczba prób RandomSearch (zadanie 3)
EPOKI_TUNER = 9                                  # epoki na próbę tunera; max_epochs Hyperbandu (zadania rozszerzone)
PATIENCE_ES = 4                                  # cierpliwość EarlyStopping (epoki)
PATIENCE_LR = 2                                  # cierpliwość ReduceLROnPlateau (epoki)
N_PETS = 200                                     # zdjęcia Oxford-IIIT Pet do Grad-CAM (zadanie 5, część B)
KATALOG_TUNERA = "lab04_tuner"                   # katalog roboczy KerasTunera (usuwany na końcu notebooka)

In [ ]:
import gc
import io
import os
import shutil
import time

import keras_tuner as kt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from IPython import display as ipd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)
T_START = time.perf_counter()
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 80

tf.get_logger().setLevel("ERROR")
print("Keras", keras.__version__, "| backend", keras.backend.backend(), "| TensorFlow", tf.__version__,
      "| KerasTuner", kt.__version__)
gpu = tf.config.list_physical_devices("GPU")
print("GPU:", gpu if gpu else "brak, obliczenia na CPU")

In [ ]:
# Wczytanie CIFAR-10 (przy pierwszym uruchomieniu pobiera około 170 MB) i podział jak w laboratorium 2.
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train_full = y_train_full.ravel()
y_test = y_test.ravel()
KLASY = ["samolot", "samochód", "ptak", "kot", "jeleń", "pies", "żaba", "koń", "statek", "ciężarówka"]

x_tr = x_train_full[:N_TRAIN].astype("float32") / 255.0
y_tr = y_train_full[:N_TRAIN]
x_val = x_train_full[-N_VAL:].astype("float32") / 255.0
y_val = y_train_full[-N_VAL:]
x_te = x_test.astype("float32") / 255.0      # test odłożony do zadania 4, liczony raz

for nazwa, y in [("trening", y_tr), ("walidacja", y_val), ("test", y_test)]:
    licz = np.bincount(y, minlength=10)
    print(f"{nazwa:10s} {len(y):6d} obrazów, na klasę od {licz.min()} do {licz.max()}")

In [ ]:
# Gotowe funkcje pomocnicze

def zbuduj_cnn(filtry=32, dropout=0.25, nazwa="cnn"):
    """Mała CNN w stylu laboratorium 2 (bez kompilacji). Ostatnia warstwa splotowa nazywa się "conv_ost"
    i daje mapy cech 8x8, których użyjemy w Grad-CAM."""
    wejscie = keras.Input(shape=(32, 32, 3))
    x = layers.Conv2D(filtry, 3, padding="same", activation="relu", name="conv1")(wejscie)
    x = layers.MaxPooling2D(2, name="pool1")(x)
    x = layers.Conv2D(2 * filtry, 3, padding="same", activation="relu", name="conv2")(x)
    x = layers.MaxPooling2D(2, name="pool2")(x)
    x = layers.Conv2D(4 * filtry, 3, padding="same", activation="relu", name="conv_ost")(x)
    x = layers.MaxPooling2D(2, name="pool3")(x)
    x = layers.Flatten(name="flatten")(x)
    x = layers.Dropout(dropout, name="dropout")(x)
    wyjscie = layers.Dense(10, activation="softmax", name="wyjscie")(x)
    return keras.Model(wejscie, wyjscie, name=nazwa)


def kompiluj(model, optymalizator):
    """Argumenty compile zawsze z nazwami: pozycyjnie trzeci argument to loss_weights, nie metrics."""
    model.compile(optimizer=optymalizator, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def podsumuj(nazwa, historia, czas_s):
    """Jeden wiersz tabeli wyników na podstawie historii treningu."""
    h = historia.history
    return {"wariant": nazwa, "epoki": len(h["loss"]),
            "train_acc (ost.)": round(h["accuracy"][-1], 4),
            "val_acc (ost.)": round(h["val_accuracy"][-1], 4),
            "val_acc (maks.)": round(max(h["val_accuracy"]), 4),
            "min val_loss": round(min(h["val_loss"]), 4),
            "epoka min val_loss": int(np.argmin(h["val_loss"]) + 1),
            "s/epokę": round(czas_s / len(h["loss"]), 1)}


def rysuj_krzywe(historie, metryki=("loss", "accuracy"), tylko_walidacja=False, tytul=None):
    """historie: słownik nazwa -> History. Linia przerywana: trening, ciągła: walidacja."""
    fig, osie = plt.subplots(1, len(metryki), figsize=(6 * len(metryki), 4), squeeze=False)
    for ax, m in zip(osie.ravel(), metryki):
        for i, (nazwa, h) in enumerate(historie.items()):
            h = h.history if hasattr(h, "history") else h
            epoki = np.arange(1, len(h[m]) + 1)
            if not tylko_walidacja:
                ax.plot(epoki, h[m], "--", color=f"C{i}", alpha=0.6, label=f"{nazwa}: trening")
            ax.plot(epoki, h["val_" + m], "o-", ms=3, color=f"C{i}", label=f"{nazwa}: walidacja")
        ax.set_xlabel("epoka")
        ax.set_ylabel(m)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    if tytul:
        fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()


def rysuj_przebieg(historia, epoka_najlepsza=None, epoka_stopu=None, zmiany_lr=(), tytul=None):
    """Strata, dokładność i learning rate w funkcji epoki z zaznaczoną najlepszą epoką, epoką zatrzymania
    i epokami, od których obowiązuje nowy learning rate. Epoki numerujemy od 1."""
    h = historia.history if hasattr(historia, "history") else historia
    epoki = np.arange(1, len(h["loss"]) + 1)
    ma_lr = "learning_rate" in h
    fig, osie = plt.subplots(1, 3 if ma_lr else 2, figsize=(16 if ma_lr else 11, 4))
    for ax, (m, opis) in zip(osie, [("loss", "strata"), ("accuracy", "dokładność")]):
        ax.plot(epoki, h[m], "o--", ms=3, alpha=0.6, label="trening")
        ax.plot(epoki, h["val_" + m], "o-", ms=3, label="walidacja")
        ax.set_ylabel(opis)
    if ma_lr:
        osie[2].semilogy(epoki, h["learning_rate"], "o-", ms=3, color="C2", label="learning rate")
        osie[2].set_ylabel("learning rate (skala log.)")
    for ax in osie:
        if epoka_najlepsza:
            ax.axvline(epoka_najlepsza, color="green", ls=":", lw=2, label=f"najlepsza epoka ({epoka_najlepsza})")
        if epoka_stopu:
            ax.axvline(epoka_stopu, color="red", ls="--", label=f"zatrzymanie ({epoka_stopu})")
        for j, e in enumerate(zmiany_lr):
            ax.axvline(e, color="gray", ls="-.", alpha=0.6, label="zmiana learning rate" if j == 0 else None)
        ax.set_xlabel("epoka")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    if tytul:
        fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()


def pokaz_siatke(obrazy, podpisy, n_kol=5, tytul=None):
    """Siatka obrazów z podpisami (podpisy mogą mieć kilka linii)."""
    n_wier = int(np.ceil(len(obrazy) / n_kol))
    fig, osie = plt.subplots(n_wier, n_kol, figsize=(2.0 * n_kol, 2.7 * n_wier), squeeze=False)
    for ax in osie.ravel():
        ax.axis("off")
    for ax, obr, p in zip(osie.ravel(), obrazy, podpisy):
        ax.imshow(obr)
        ax.set_title(p, fontsize=8)
    if tytul:
        fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()


def rysuj_macierz(cm, nazwy, tytul=None):
    """Macierz pomyłek: wiersze to klasy prawdziwe, kolumny przewidziane."""
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(cm, cmap="Blues")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(len(nazwy)))
    ax.set_xticklabels(nazwy, rotation=45, ha="right")
    ax.set_yticks(range(len(nazwy)))
    ax.set_yticklabels(nazwy)
    ax.set_xlabel("klasa przewidziana")
    ax.set_ylabel("klasa prawdziwa")
    fig.colorbar(im, ax=ax, fraction=0.046)
    if tytul:
        ax.set_title(tytul)
    plt.tight_layout()
    plt.show()


class LicznikEpok(keras.callbacks.Callback):
    """Liczy epoki we wszystkich próbach tunera. KerasTuner kopiuje callbacki (deepcopy) przed każdą próbą,
    dlatego licznik jest atrybutem klasy, a nie obiektu. Przed każdym przeszukiwaniem: LicznikEpok.epoki = 0."""
    epoki = 0

    def on_epoch_end(self, epoch, logs=None):
        LicznikEpok.epoki += 1


def tabela_prob(tuner):
    """Tabela wszystkich zakończonych prób tunera, od najlepszej. Wynik próby to najlepsza val_accuracy
    spośród jej epok (tak KerasTuner liczy wynik dla objective="val_accuracy")."""
    wiersze = []
    for t in tuner.oracle.trials.values():
        if t.score is None:
            continue
        hp = {k: v for k, v in t.hyperparameters.values.items() if k != "tuner/trial_id"}
        wiersze.append({"próba": t.trial_id, **hp, "val_accuracy (najl.)": round(t.score, 4)})
    return pd.DataFrame(wiersze).sort_values("val_accuracy (najl.)", ascending=False).reset_index(drop=True)

## Zadanie 1: Callbacki

1. Zbudujcie model `zbuduj_cnn(filtry=32, dropout=0.25)` i skompilujcie go z Adamem (learning rate 1e-3).
2. Przygotujcie trzy callbacki: `EarlyStopping(monitor="val_loss", patience=PATIENCE_ES, restore_best_weights=True)`, `ModelCheckpoint` zapisujący najlepszy model do pliku `.keras` w bieżącym katalogu (`save_best_only=True`) i `ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=PATIENCE_LR, min_lr=1e-5)`. Dobrze jest napisać funkcję, która za każdym razem tworzy nowe obiekty, bo przyda się w zadaniu 3.
3. Trenujcie najwyżej `EPOKI_MAX` epok z walidacją. Zbiór testowy zostaje nietknięty.
4. Wyznaczcie najlepszą epokę (minimum `val_loss`), epokę zatrzymania i epoki zmiany learning rate (klucz `learning_rate` w historii). Narysujcie przebieg funkcją `rysuj_przebieg`.
5. Sprawdźcie, że model w pamięci po treningu i model wczytany z pliku checkpointu dają tę samą stratę walidacyjną co najlepsza epoka.

Pytania: W której epoce trening się zatrzymał i dlaczego właśnie wtedy? Jakie wagi ma model po treningu? Co działo się z krzywymi po każdej redukcji learning rate? Co by się stało, gdyby EarlyStopping monitorował `accuracy` zamiast `val_loss`? Dlaczego `patience` w ReduceLROnPlateau powinna być mniejsza niż w EarlyStopping?

In [ ]:
SCIEZKA_CKPT = "lab04_cnn_najlepszy.keras"

# TODO: funkcja nowe_callbacki(checkpoint=None), która zwraca listę nowych obiektów:
#   EarlyStopping(monitor="val_loss", patience=PATIENCE_ES, restore_best_weights=True, verbose=1),
#   ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=PATIENCE_LR, min_lr=1e-5, verbose=1),
#   oraz ModelCheckpoint(checkpoint, monitor="val_loss", save_best_only=True), jeśli podano ścieżkę.
# TODO: keras.utils.set_random_seed(RANDOM_STATE); model_cb = kompiluj(zbuduj_cnn(32, 0.25), Adam(1e-3));
#   hist_cb = model_cb.fit(x_tr, y_tr, validation_data=(x_val, y_val), epochs=EPOKI_MAX, batch_size=BATCH,
#                          callbacks=callbacki, verbose=2); zmierzcie czas treningu.
model_cb, hist_cb, callbacki = None, None, None

In [ ]:
# TODO: z hist_cb.history wyznaczcie epoka_najl (np.argmin(val_loss) + 1), epoka_stopu
#   (callbacki[0].stopped_epoch + 1, jeśli > 0, inaczej None) i zmiany_lr (epoki, od których learning rate
#   z klucza "learning_rate" jest inny niż w poprzedniej epoce). Narysujcie:
#   rysuj_przebieg(hist_cb, epoka_najl, epoka_stopu, zmiany_lr, tytul=...)
# TODO: porównajcie min(val_loss) z model_cb.evaluate(x_val, y_val) i z modelem wczytanym
#   keras.models.load_model(SCIEZKA_CKPT).

Odpowiedzi:

1. 
2. 
3. 
4. 
5. 

## Zadanie 2: Optymalizatory

1. Porównajcie trzy optymalizatory przy tej samej architekturze (`zbuduj_cnn(32, 0.25)`), tych samych wagach początkowych (to samo ziarno przed budową modelu) i tym samym budżecie `EPOKI_OPT` epok, bez callbacków: SGD bez momentum (learning rate 0,01), SGD z momentum 0,9 (learning rate 0,01) i Adam (learning rate 0,001).
2. Zbudujcie tabelę wyników (`podsumuj`) i wspólny wykres `val_loss` i `val_accuracy` trzech wariantów.
3. Wytrenujcie jeszcze raz model z Adamem, ale z learning rate 0,03, i porównajcie jego krzywe z Adamem 0,001.

Pytania: Który optymalizator najszybciej obniżył stratę walidacyjną i o ile różnią się wyniki po `EPOKI_OPT` epokach? Co daje momentum w porównaniu ze zwykłym SGD? Czy wynik tego porównania przesądza, że Adam jest lepszy od SGD z momentum? Co dzieje się przy za dużym learning rate i po czym to poznać w historii treningu?

In [ ]:
# TODO: słownik OPTYMALIZATORY nazwa -> funkcja tworząca nowy optymalizator:
#   SGD(learning_rate=0.01), SGD(learning_rate=0.01, momentum=0.9), Adam(learning_rate=1e-3).
# Dla każdego: keras.utils.set_random_seed(RANDOM_STATE), kompiluj(zbuduj_cnn(32, 0.25), utworz()),
#   fit przez EPOKI_OPT epok z walidacją (verbose=0), zapis historii w historie_opt i wiersza podsumuj(...)
#   w wiersze_opt.
historie_opt, wiersze_opt = {}, []

In [ ]:
# TODO: rysuj_krzywe(historie_opt, tylko_walidacja=True, tytul=...) i tabela pd.DataFrame(wiersze_opt).

In [ ]:
# TODO: ten sam model z Adam(learning_rate=0.03), EPOKI_OPT epok, verbose=0. Wypiszcie loss i val_accuracy
#   w każdej epoce i narysujcie krzywe obok Adama z learning rate 0,001 (rysuj_krzywe).
hist_lr = None

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 3: KerasTuner

1. Napiszcie `zbuduj_model_hp(hp)`, która przeszukuje trzy hiperparametry: liczbę filtrów (`hp.Choice("filtry", [16, 32, 64])`), dropout (`hp.Choice("dropout", [0.0, 0.25, 0.5])`) i learning rate Adama (`hp.Choice("learning_rate", [3e-4, 1e-3, 3e-3])`), i zwraca skompilowany model `zbuduj_cnn`.
2. Utwórzcie `kt.RandomSearch` z `objective="val_accuracy"`, `max_trials=MAX_PROB`, `seed=RANDOM_STATE`, `directory=KATALOG_TUNERA`, `project_name="random_search"`, `overwrite=True` i uruchomcie `search` na zbiorze treningowym z walidacją, po `EPOKI_TUNER` epok na próbę. Przekażcie `callbacks=[LicznikEpok()]` (wcześniej `LicznikEpok.epoki = 0`), żeby policzyć zużyte epoki. Zmierzcie czas.
3. Pokażcie tabelę prób (`tabela_prob`).
4. Weźcie najlepszą konfigurację (`get_best_hyperparameters`), wytrenujcie ją od nowa z callbackami z zadania 1 (bez checkpointu) przez najwyżej `EPOKI_MAX` epok i porównajcie na walidacji z modelem z zadania 1. Wybierzcie lepszy model według `val_accuracy`. Testu nadal nie ruszacie.

Pytania: Ile konfiguracji ze wszystkich możliwych sprawdził tuner i ile epok to kosztowało? Które hiperparametry najbardziej różnicują wyniki? Dlaczego najlepszy model z tunera trenujemy jeszcze raz, zamiast od razu go użyć? Na czym tuner opierał wybór i dlaczego nie może to być zbiór testowy? Czy różnica między dwiema najlepszymi próbami jest wiarygodna (wrócicie do tego w zadaniach rozszerzonych)?

In [ ]:
# TODO: zbuduj_model_hp(hp): filtry = hp.Choice("filtry", [16, 32, 64]), dropout = hp.Choice("dropout",
#   [0.0, 0.25, 0.5]), lr = hp.Choice("learning_rate", [3e-4, 1e-3, 3e-3]);
#   return kompiluj(zbuduj_cnn(filtry, dropout), keras.optimizers.Adam(learning_rate=lr)).
# TODO: tuner_rs = kt.RandomSearch(zbuduj_model_hp, objective="val_accuracy", max_trials=MAX_PROB,
#   seed=RANDOM_STATE, directory=KATALOG_TUNERA, project_name="random_search", overwrite=True)
#   LicznikEpok.epoki = 0; tuner_rs.search(x_tr, y_tr, validation_data=(x_val, y_val), epochs=EPOKI_TUNER,
#   batch_size=BATCH, callbacks=[LicznikEpok()], verbose=0); zapiszcie czas_rs i epoki_rs = LicznikEpok.epoki.
tuner_rs = None

In [ ]:
# TODO: tabela_rs = tabela_prob(tuner_rs)
tabela_rs = None

In [ ]:
# TODO: najlepsze_hp = tuner_rs.get_best_hyperparameters(1)[0]; ziarno; model_tuner = zbuduj_model_hp(najlepsze_hp);
#   fit przez EPOKI_MAX epok z callbacks=nowe_callbacki() (bez checkpointu), verbose=2.
# TODO: porównajcie model_cb i model_tuner na walidacji (evaluate na x_val), zbudujcie tabelę i wybierzcie
#   model_wybrany według val_accuracy. Testu jeszcze nie liczycie.
najlepsze_hp, model_tuner, model_wybrany = None, None, None

Odpowiedzi:

1. 
2. 
3. 
4. 
5. 

## Zadanie 4: Ewaluacja na zbiorze testowym

1. Dla wybranego modelu policzcie raz predykcje na zbiorze testowym (`predict`). Wypiszcie dokładność i macro F1 (`f1_score(..., average="macro")`).
2. Narysujcie macierz pomyłek (`confusion_matrix`, `rysuj_macierz`) i wypiszcie pięć najczęściej mylonych par klas.
3. Wypiszcie `classification_report` z nazwami klas.
4. Pokażcie dziesięć najpewniejszych błędów: obrazy, na których model się pomylił, posortowane według prawdopodobieństwa przypisanego błędnej klasie.
5. Zbudujcie z wyników testu zbiór niezbalansowany: wszystkie obrazy klasy „samochód” i po 50 obrazów każdej innej klasy. Porównajcie dokładność i macro F1 waszego modelu oraz klasyfikatora, który zawsze odpowiada „samochód”.

Pytania: Które klasy model myli najczęściej i czy te pomyłki są zrozumiałe? Dlaczego na pełnym zbiorze testowym dokładność i macro F1 są prawie równe, a na zbiorze niezbalansowanym nie? Co mówi precision, a co recall dla klasy „kot”? Co widać na najpewniejszych błędach (etykieta, tło, podobieństwo klas)? Dlaczego test liczymy tylko raz i dopiero teraz?

In [ ]:
# TODO: p_te = model_wybrany.predict(x_te, batch_size=256, verbose=0); y_pred = p_te.argmax(axis=1);
#   accuracy_score i f1_score(..., average="macro"); cm = confusion_matrix(y_test, y_pred);
#   rysuj_macierz(cm, KLASY, tytul=...); pięć największych elementów poza przekątną.
p_te, y_pred = None, None

In [ ]:
# TODO: print(classification_report(y_test, y_pred, target_names=KLASY, digits=3, zero_division=0))

In [ ]:
# TODO: indeksy błędów (y_pred != y_test), prawdopodobieństwo błędnej klasy p_te[i, y_pred[i]],
#   dziesięć największych (idx_bledy_pewne) i siatka pokaz_siatke(x_test[idx], podpisy, n_kol=5) z podpisami
#   "prawda: ... / model: ... (p)".
idx_bledy_pewne = None

In [ ]:
# TODO: zbiór niezbalansowany z wyników testu (bez ponownego predict): wszystkie indeksy klasy "samochód"
#   i po 50 losowych indeksów (rng = np.random.default_rng(RANDOM_STATE)) każdej innej klasy.
#   Accuracy i macro F1 dla y_pred oraz dla stałej odpowiedzi "samochód" (f1_score z zero_division=0).

Odpowiedzi:

1. 
2. 
3. 
4. 
5. 

## Zadanie 5: Grad-CAM

Grad-CAM (Selvaraju i in., 2017) pokazuje, które miejsca obrazu najbardziej podniosły wynik wybranej klasy. Bierzemy mapy cech $A^k$ ostatniej warstwy splotowej (kanały $k$) i wynik klasy przed softmaxem (logit $y^c$). Waga kanału to średni gradient $\alpha_k = \frac{1}{Z}\sum_{i,j} \partial y^c / \partial A^k_{ij}$, a mapa to $\mathrm{ReLU}(\sum_k \alpha_k A^k)$, przeskalowana do rozmiaru obrazu.

Gradient liczymy przez `tf.GradientTape`, więc to jest kod dla backendu TensorFlow (na innym backendzie trzeba by użyć jego mechanizmu gradientów). Gotowa funkcja `podziel_model` przechodzi kolejno przez warstwy modelu do wskazanej warstwy (mapy cech), potem przez resztę głowy, a z ostatniej warstwy `Dense` liczy logity zamiast prawdopodobieństw. Gotowa jest też funkcja `pokaz_gradcam`.

1. Napiszcie `grad_cam(przejscie, obraz, klasa=None)`: w `GradientTape` wywołajcie `przejscie` na obrazie z dodaną osią wsadu, weźcie logit klasy (domyślnie przewidzianej), policzcie gradient po mapach cech, uśrednijcie go po wysokości i szerokości, zważcie mapy, zsumujcie po kanałach, przepuśćcie przez ReLU i podzielcie przez maksimum.
2. Część A (własna CNN z CIFAR-10, wykonujecie zawsze; bez pliku z laboratorium 3 to jedyna część): `podziel_model(model_wybrany, "conv_ost")`. Sprawdźcie, że softmax z logitów daje te same prawdopodobieństwa co `predict`. Pokażcie Grad-CAM dla czterech losowych trafień i czterech najpewniejszych błędów z zadania 4.
3. Część B (MobileNetV2 z laboratorium 3): gotowa komórka szuka pliku `lab03_pets.keras` i jeśli go znajdzie, wczytuje model oraz `N_PETS` zdjęć testowych Oxford-IIIT Pet. Baza to warstwa `mobilenetv2_1.00_160`, a jej ostatnia warstwa to `out_relu` (mapy 5x5x1280), więc `podziel_model(model_pets, "mobilenetv2_1.00_160")`. Sprawdźcie zgodność predykcji, pokażcie Grad-CAM dla czterech trafień, a dla czterech najpewniejszych błędów mapy dla klasy przewidzianej i dla prawdziwej.

Pytania: Na co patrzy własna CNN przy trafieniach, a na co przy błędach? Dlaczego mapy dla CIFAR-10 są tak rozmyte i co ogranicza ich rozdzielczość w obu modelach? Czym różnią się mapy dla klasy przewidzianej i prawdziwej przy błędach modelu z laboratorium 3? Dlaczego używamy logitu, a nie prawdopodobieństwa po softmaxie? Czego Grad-CAM nie mówi o działaniu sieci?

In [ ]:
def podziel_model(model, nazwa_warstwy_map):
    """Zwraca funkcję przejscie(x) -> (mapy cech, logity). Przechodzi kolejno przez warstwy najwyższego
    poziomu modelu (bez rozgałęzień) aż do warstwy nazwa_warstwy_map włącznie, potem przez resztę głowy.
    Z ostatniej warstwy Dense bierzemy logity (x @ kernel + bias), czyli wynik przed softmaxem.
    Warstwy są wywoływane z training=False (augmentacja i dropout nieaktywne, BatchNormalization w trybie predykcji).
    Uwaga: jeśli warstwą map jest zagnieżdżony model (np. baza MobileNetV2), mapy cech to jego wyjście."""
    warstwy = [w for w in model.layers if not isinstance(w, keras.layers.InputLayer)]
    i = [w.name for w in warstwy].index(nazwa_warstwy_map)
    do_map, glowa, wyjscie = warstwy[: i + 1], warstwy[i + 1: -1], warstwy[-1]

    def przejscie(x):
        for w in do_map:
            x = w(x, training=False)
        mapy = x
        for w in glowa:
            x = w(x, training=False)
        logity = keras.ops.matmul(x, wyjscie.kernel) + wyjscie.bias
        return mapy, logity

    return przejscie


def pokaz_gradcam(obrazy, mapy, podpisy, n_kol=4, tytul=None):
    """Dla każdego obrazu dwa panele: oryginał i mapa Grad-CAM nałożona na obraz (rząd niżej).
    Obrazy uint8 (0..255) albo float w [0, 1]; mapy w [0, 1] o dowolnej rozdzielczości."""
    n_wier = int(np.ceil(len(obrazy) / n_kol))
    fig, osie = plt.subplots(2 * n_wier, n_kol, figsize=(2.4 * n_kol, 2.6 * 2 * n_wier), squeeze=False)
    for ax in osie.ravel():
        ax.axis("off")
    for i, (obr, cam, p) in enumerate(zip(obrazy, mapy, podpisy)):
        w, k = 2 * (i // n_kol), i % n_kol
        obr = obr.astype("float32") / 255.0 if obr.dtype == np.uint8 else obr
        cam_duza = tf.image.resize(np.asarray(cam)[..., None], obr.shape[:2], method="bilinear").numpy()[..., 0]
        osie[w, k].imshow(obr)
        osie[w, k].set_title(p, fontsize=8)
        osie[w + 1, k].imshow(obr)
        osie[w + 1, k].imshow(cam_duza, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    if tytul:
        fig.suptitle(tytul)
    plt.tight_layout()
    # Zdjęcia z mapami ciepła zapisane jako PNG ważą kilkaset kB, więc wyświetlamy JPEG (mniejszy notebook).
    bufor = io.BytesIO()
    fig.savefig(bufor, format="jpeg", dpi=80, pil_kwargs={"quality": 80})
    plt.close(fig)
    ipd.display(ipd.Image(data=bufor.getvalue(), format="jpeg"))

In [ ]:
def grad_cam(przejscie, obraz, klasa=None):
    """Mapa Grad-CAM (wysokość i szerokość map cech, wartości w [0, 1]) i numer klasy. Kod dla backendu TensorFlow."""
    # TODO: x = tf.convert_to_tensor(obraz[None], dtype=tf.float32)
    #   with tf.GradientTape() as tape: mapy, logity = przejscie(x); klasa = argmax logitów, jeśli None;
    #       wynik = logity[:, klasa]
    #   gradienty = tape.gradient(wynik, mapy); wagi = średnia gradientów po osiach (1, 2);
    #   cam = ReLU(suma po kanałach mapy * wagi), podzielona przez maksimum (+ 1e-8).
    return None, klasa

### Część A: własna CNN z CIFAR-10

In [ ]:
# TODO: przejscie_cifar = podziel_model(model_wybrany, "conv_ost"); sprawdźcie na x_te[:64], że
#   keras.ops.softmax(logity) daje to samo co p_te[:64] (maksymalna różnica bezwzględna).
# TODO: 4 losowe trafienia (rng = np.random.default_rng(RANDOM_STATE)) i 4 pierwsze z idx_bledy_pewne;
#   dla każdego grad_cam(przejscie_cifar, x_te[i]) i podpis z klasą prawdziwą i przewidzianą;
#   pokaz_gradcam(x_te[idx], mapy, podpisy, n_kol=4, tytul=...).
przejscie_cifar = None

### Część B: model z laboratorium 3

W Colabie wgrajcie plik `lab03_pets.keras` (około 24 MB) do bieżącego katalogu: panel Pliki po lewej stronie, Prześlij do magazynu sesji. Jeśli zapisaliście go na Dysku Google, zamontujcie Dysk i skopiujcie plik (przykład w komentarzu w komórce). Bez pliku komórka wypisze komunikat, a Grad-CAM wykonujecie tylko w części A. Model zawiera warstwę `Rescaling`, więc przyjmuje piksele 0..255, tak jak w laboratorium 3.

In [ ]:
# W Colabie, jeśli plik jest na Dysku Google:
#   from google.colab import drive
#   drive.mount("/content/drive")
#   !cp "/content/drive/MyDrive/<wasz katalog>/lab03_pets.keras" .
KANDYDACI_PETS = ["lab03_pets.keras", os.path.join("..", "lab03_transfer_learning", "lab03_pets.keras")]
URL_PETS_TEST = "https://huggingface.co/datasets/timm/oxford-iiit-pet/resolve/main/data/test-00000-of-00001.parquet"
KLASY_PETS = [
    "abyssinian", "american_bulldog", "american_pit_bull_terrier", "basset_hound", "beagle",
    "bengal", "birman", "bombay", "boxer", "british_shorthair", "chihuahua", "egyptian_mau",
    "english_cocker_spaniel", "english_setter", "german_shorthaired", "great_pyrenees", "havanese",
    "japanese_chin", "keeshond", "leonberger", "maine_coon", "miniature_pinscher", "newfoundland",
    "persian", "pomeranian", "pug", "ragdoll", "russian_blue", "saint_bernard", "samoyed",
    "scottish_terrier", "shiba_inu", "siamese", "sphynx", "staffordshire_bull_terrier",
    "wheaten_terrier", "yorkshire_terrier",
]


def wczytaj_pets_probke(n, rozmiar=160):
    """Pobiera część testową Oxford-IIIT Pet (około 410 MB), losuje n zdjęć, skaluje je do 160x160
    jak w laboratorium 3 i zwraca X (n, 160, 160, 3) uint8 oraz etykiety y."""
    df = pd.read_parquet(URL_PETS_TEST, columns=["image", "label"])
    idx = np.sort(np.random.default_rng(RANDOM_STATE).choice(len(df), size=n, replace=False))
    X = np.empty((n, rozmiar, rozmiar, 3), dtype=np.uint8)
    for j, i in enumerate(idx):
        with Image.open(io.BytesIO(df["image"].iloc[i]["bytes"])) as img:
            X[j] = np.asarray(img.convert("RGB").resize((rozmiar, rozmiar), Image.BILINEAR))
    y = df["label"].to_numpy()[idx].astype("int64")
    del df
    gc.collect()
    return X, y


sciezka_pets = next((p for p in KANDYDACI_PETS if os.path.exists(p)), None)
model_pets, X_pets, y_pets = None, None, None
if sciezka_pets is None:
    print("Nie znaleziono lab03_pets.keras. Część B pomijacie, Grad-CAM tylko w części A.")
else:
    model_pets = keras.models.load_model(sciezka_pets)
    print("Wczytano model z laboratorium 3. Warstwy najwyższego poziomu:")
    for w in model_pets.layers:
        print(f"  {w.name:28s} {type(w).__name__:24s} {tuple(w.output.shape)}")
    t0 = time.perf_counter()
    X_pets, y_pets = wczytaj_pets_probke(N_PETS)
    print(f"Próbka Oxford-IIIT Pet: {X_pets.shape} {X_pets.dtype}, wczytana w {time.perf_counter() - t0:.0f} s")

In [ ]:
if model_pets is None:
    print("Brak modelu z laboratorium 3, część B pominięta.")
else:
    # TODO: przejscie_pets = podziel_model(model_pets, "mobilenetv2_1.00_160"); p_pets = model_pets.predict(X_pets);
    #   sprawdźcie zgodność softmax(logity) z p_pets na X_pets[:32] i kształt map cech (out_relu).
    # TODO: trafność na próbce; 4 losowe trafienia i 4 najpewniejsze błędy; Grad-CAM dla trafień;
    #   dla błędów mapy dla klasy przewidzianej i dla prawdziwej (klasa=int(y_pets[i])), podpisy z KLASY_PETS.
    pass

Odpowiedzi:

1. 
2. 
3. 
4. 
5. 

# Zadania rozszerzone

## Zadanie 6: Hyperband a RandomSearch

1. Utwórzcie `kt.Hyperband` z tą samą funkcją `zbuduj_model_hp`, `objective="val_accuracy"`, `max_epochs=EPOKI_TUNER`, `factor=3`, `hyperband_iterations=1`, `seed=RANDOM_STATE`, w tym samym katalogu z `project_name="hyperband"` i `overwrite=True`. Uruchomcie `search` z `LicznikEpok` (liczbę epok wyznacza tu Hyperband, nie podajecie `epochs`).
2. Pokażcie tabelę prób. Kolumny `tuner/epochs` i `tuner/round` mówią, do której epoki trenowano próbę i w której rundzie.
3. Zbudujcie tabelę porównawczą obu tunerów: liczba prób, liczba różnych konfiguracji, łączna liczba epok, czas, najlepsza `val_accuracy` i najlepsza konfiguracja.

Pytania: Czy budżety obu przeszukiwań są porównywalne w epokach? Ile konfiguracji sprawdził każdy tuner i ile z nich Hyperband trenował przez pełne `max_epochs`? Czy oba wskazały tę samą konfigurację? Na czym polega ryzyko odrzucania konfiguracji po jednej lub dwóch epokach?

In [ ]:
# TODO: tuner_hb = kt.Hyperband(zbuduj_model_hp, objective="val_accuracy", max_epochs=EPOKI_TUNER, factor=3,
#   hyperband_iterations=1, seed=RANDOM_STATE, directory=KATALOG_TUNERA, project_name="hyperband", overwrite=True)
#   LicznikEpok.epoki = 0; search bez epochs, z callbacks=[LicznikEpok()], verbose=0; czas_hb, epoki_hb;
#   tabela_hb = tabela_prob(tuner_hb).
tuner_hb = None

In [ ]:
# TODO: tabela porównawcza obu tunerów: liczba prób (tuner.oracle.trials), liczba różnych konfiguracji
#   (hiperparametry bez kluczy zaczynających się od "tuner/"), liczba prób trenowanych do pełnych EPOKI_TUNER,
#   epoki łącznie, czas, najlepsza val_accuracy (tuner.oracle.get_best_trials(1)[0].score) i konfiguracja.

Odpowiedzi:

1. 
2. 
3. 
4. 

## Zadanie 7: Zmienność wyniku między seedami

1. Wytrenujcie najlepszą konfigurację z RandomSearch (`najlepsze_hp`) trzy razy z ziarnami 42, 43 i 44, za każdym razem przez `EPOKI_TUNER` epok, bez callbacków, czyli tak samo jak w próbie tunera.
2. Dla każdego ziarna zapiszcie najlepszą `val_accuracy` spośród epok (tak liczy wynik KerasTuner) i `val_accuracy` po ostatniej epoce. Policzcie średnią i odchylenie standardowe (z poprawką na próbę, `ddof=1`). Narysujcie krzywe `val_accuracy` trzech przebiegów.
3. Porównajcie odchylenie z różnicami między próbami w tabeli RandomSearch: między pierwszą a drugą próbą i między pierwszą a medianą.

Pytania: Jak duża jest zmienność wyniku przy tej samej konfiguracji? Czy różnice między najlepszymi konfiguracjami z tunera są od niej wyraźnie większe? Co z tego wynika dla wyboru zwycięzcy na podstawie jednej próby? Jak zmienilibyście procedurę tuningu, mając więcej czasu obliczeniowego (na przykład `executions_per_trial`)?

In [ ]:
# TODO: dla s w [42, 43, 44]: keras.utils.set_random_seed(s); m = zbuduj_model_hp(najlepsze_hp);
#   fit przez EPOKI_TUNER epok (verbose=0); zapiszcie historię i wiersz z max(val_accuracy) i ostatnią val_accuracy.
# TODO: średnia i odchylenie standardowe (ddof=1); rysuj_krzywe(historie_seed, metryki=("accuracy",),
#   tylko_walidacja=True).
tabela_seed = None

In [ ]:
# TODO: z tabela_rs["val_accuracy (najl.)"] policzcie różnicę między 1. a 2. próbą i między 1. próbą a medianą,
#   wyraźcie je w odchyleniach standardowych z poprzedniej komórki i porównajcie wynik najlepszej próby
#   ze średnią z trzech seedów.

Odpowiedzi:

1. 
2. 
3. 
4. 

In [ ]:
print(f"Czas wykonania notebooka (od komórki importów): {(time.perf_counter() - T_START) / 60:.1f} min")

In [ ]:
# Sprzątanie: katalog tunera zawiera wagi z każdej próby i nie jest już potrzebny.
# Plik checkpointu (SCIEZKA_CKPT) zostaje, możecie go pobrać.
shutil.rmtree(KATALOG_TUNERA, ignore_errors=True)
print("Usunięto katalog", KATALOG_TUNERA)